# Phase 8: Customer Intelligence Dashboard

### Objective

Bring together the outputs of every previous phase — cleaning, feature engineering, RFM segmentation, and churn prediction — into a single interactive dashboard summarizing the business picture and the customers who need attention.

### Inputs

- `customer_features.csv`
- `rfm_customer_segments_scored.csv` (from Phase 7, includes `Churn_Probability`)
- `orders_features.csv`
- `customers_clean.csv`

### Workflow

1. Load All Processed Data
2. Business KPI Summary
3. Revenue Trend Over Time
4. Customer Segment Distribution & Revenue
5. Churn Risk Overview
6. Geographic Distribution
7. High-Risk Customer Watchlist
8. Executive Summary

### 1. Load All Processed Data

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

customer_features = pd.read_csv("../data/processed/customer_features.csv")
rfm = pd.read_csv("../data/processed/rfm_customer_segments_scored.csv")
orders = pd.read_csv("../data/processed/orders_features.csv")
customers = pd.read_csv("../data/processed/customers_clean.csv")

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

print(f"Customers (RFM):  {len(rfm):,}")
print(f"Orders:           {len(orders):,}")
rfm.head()

Customers (RFM):  96,096
Orders:           99,441


,customer_unique_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Customer_Segment,Churn,Churn_Probability,Predicted_Churn
0,0000366f3b9a7992bf8c76cfdf3221e2,160,1,141.90,4,1,4,414,Potential Loyalists,0,0.0,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,163,1,27.19,4,1,1,411,Potential Loyalists,0,0.0,0
2,0000f46a3911fa3c0805444483337064,585,1,86.22,1,1,2,112,Lost Customers,1,1.0,1
3,0000f6ccb0745a6a4b88665a16c9f078,369,1,43.62,2,1,1,211,Lost Customers,1,1.0,1
4,0004aac84e0df4da2b147fca70cf8255,336,1,196.89,2,1,4,214,Lost Customers,1,1.0,1


### 2. Business KPI Summary

In [2]:
total_customers = len(rfm)
total_revenue = rfm["Monetary"].sum()
avg_order_value = customer_features["average_order_value"].mean()
churn_rate = rfm["Churn"].mean() * 100
high_risk_count = rfm["Predicted_Churn"].sum()

kpis = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Total Revenue (R$)",
        "Average Order Value (R$)",
        "Churn Rate (%)",
        "Predicted High-Risk Customers"
    ],
    "Value": [
        f"{total_customers:,}",
        f"{total_revenue:,.2f}",
        f"{avg_order_value:,.2f}",
        f"{churn_rate:.1f}%",
        f"{high_risk_count:,}"
    ]
})

kpis

,Metric,Value
0,Total Customers,"96,096"
1,Total Revenue (R$),"15,843,553.24"
2,Average Order Value (R$),159.81
3,Churn Rate (%),70.7%
4,Predicted High-Risk Customers,"67,963"


### 3. Revenue Trend Over Time

In [3]:
monthly = (
    orders
    .assign(month=orders["order_purchase_timestamp"].dt.to_period("M").astype(str))
    .groupby("month")["order_value"]
    .sum()
    .reset_index()
    .sort_values("month")
)

# Drop the first/last partial months if they look like data-boundary artifacts
fig = px.line(
    monthly, x="month", y="order_value",
    title="Monthly Revenue Trend",
    labels={"month": "Month", "order_value": "Revenue (R$)"},
    markers=True
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### 4. Customer Segment Distribution & Revenue

In [4]:
segment_summary = (
    rfm.groupby("Customer_Segment")
    .agg(
        Customer_Count=("customer_unique_id", "count"),
        Total_Revenue=("Monetary", "sum"),
    )
    .reset_index()
    .sort_values("Total_Revenue", ascending=False)
)
segment_summary["Pct_of_Customers"] = (
    segment_summary["Customer_Count"] / segment_summary["Customer_Count"].sum() * 100
).round(1)
segment_summary["Pct_of_Revenue"] = (
    segment_summary["Total_Revenue"] / segment_summary["Total_Revenue"].sum() * 100
).round(1)

segment_summary

,Customer_Segment,Customer_Count,Total_Revenue,Pct_of_Customers,Pct_of_Revenue
0,At Risk,22967,3823132.71,23.9,24.1
5,Potential Loyalists,23170,3807919.63,24.1,24.0
4,Need Attention,19043,2965015.09,19.8,18.7
2,Lost Customers,15463,2499350.15,16.1,15.8
3,Loyal Customers,11500,1978405.44,12.0,12.5
1,Champions,3953,769730.22,4.1,4.9


In [5]:
fig = px.bar(
    segment_summary, x="Customer_Segment", y="Customer_Count",
    color="Customer_Segment", text="Pct_of_Customers",
    title="Customers by Segment"
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

In [6]:
fig = px.bar(
    segment_summary, x="Customer_Segment", y="Total_Revenue",
    color="Customer_Segment", text="Pct_of_Revenue",
    title="Revenue by Segment",
    labels={"Total_Revenue": "Revenue (R$)"}
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

**Reading this chart:** compare the customer-count chart to the revenue chart. Segments that carry a disproportionately high share of revenue relative to their customer count (e.g. Champions, Loyal Customers) are the ones worth protecting first; segments with many customers but little revenue are lower priority for retention spend.

### 5. Churn Risk Overview

In [7]:
churn_by_segment = (
    rfm.groupby("Customer_Segment")["Predicted_Churn"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "High_Risk_Customers", "count": "Total_Customers"})
    .reset_index()
)
churn_by_segment["Risk_Rate_%"] = (
    churn_by_segment["High_Risk_Customers"] / churn_by_segment["Total_Customers"] * 100
).round(1)
churn_by_segment = churn_by_segment.sort_values("Risk_Rate_%", ascending=False)

fig = px.bar(
    churn_by_segment, x="Customer_Segment", y="Risk_Rate_%",
    color="Risk_Rate_%", color_continuous_scale="Reds",
    title="Predicted Churn Risk Rate by Segment",
    labels={"Risk_Rate_%": "% Flagged High-Risk"}
)
fig.show()

churn_by_segment

,Customer_Segment,High_Risk_Customers,Total_Customers,Risk_Rate_%
2,Lost Customers,15463,15463,100.0
4,Need Attention,19007,19043,99.8
0,At Risk,22893,22967,99.7
3,Loyal Customers,4155,11500,36.1
5,Potential Loyalists,6445,23170,27.8
1,Champions,0,3953,0.0


In [8]:
fig = px.histogram(
    rfm, x="Churn_Probability", nbins=40,
    title="Distribution of Predicted Churn Probability",
    labels={"Churn_Probability": "Predicted Churn Probability"}
)
fig.add_vline(x=0.5, line_dash="dash", line_color="red",
              annotation_text="Decision threshold (0.5)")
fig.show()

### 6. Geographic Distribution

In [9]:
customer_state = customers[["customer_unique_id", "customer_state"]].drop_duplicates("customer_unique_id")
rfm_geo = rfm.merge(customer_state, on="customer_unique_id", how="left")

state_summary = (
    rfm_geo.groupby("customer_state")
    .agg(Customers=("customer_unique_id", "count"), Revenue=("Monetary", "sum"))
    .reset_index()
    .sort_values("Revenue", ascending=False)
    .head(15)
)

fig = px.bar(
    state_summary, x="customer_state", y="Revenue",
    title="Top 15 States by Revenue",
    labels={"customer_state": "State", "Revenue": "Revenue (R$)"}
)
fig.show()

### 7. High-Risk Customer Watchlist

In [10]:
watchlist = (
    rfm[rfm["Predicted_Churn"] == 1]
    .sort_values(["Monetary", "Churn_Probability"], ascending=[False, False])
    .head(20)
    [["customer_unique_id", "Customer_Segment", "Recency", "Frequency", "Monetary", "Churn_Probability"]]
)

print("Top 20 highest-value customers currently flagged as high churn risk:")
watchlist

Top 20 highest-value customers currently flagged as high churn risk:


,customer_unique_id,Customer_Segment,Recency,Frequency,Monetary,Churn_Probability
3826,0a0a92112bd4c708ca5fde585afaa872,Lost Customers,383,1,13664.08,1.0
81962,da122df9eeddfedc1dc1f5349a1a690c,At Risk,564,2,7571.63,1.0
82808,dc4802a71eae9be1dd28f5d788ceb526,At Risk,611,1,6929.31,1.0
95806,ff4159b92c40ebe40454e3e6a7c35ed6,At Risk,510,1,6726.66,1.0
24121,4007669dec559734d6f53e029e360987,Lost Customers,327,1,6081.54,1.0
89688,eebb5dda148d3893cdaf5b5ca3040ccb,At Risk,546,1,4764.34,1.0
89420,edf81e1f3070b9dac83ec83dacdbb9bc,At Risk,546,1,4194.76,1.0
93998,fa562ef24d41361e476e748681810e1e,Loyal Customers,202,1,4175.26,1.0
2032,055ec572ac7f3c7bdd04a183830ebe59,At Risk,455,2,4053.08,1.0
35581,5e713be0853d8986528d7869a0811d2b,Lost Customers,619,1,4042.74,1.0


These are the highest-value customers currently flagged as likely to churn — the customers where a retention offer or outreach would protect the most revenue per dollar spent.

### 8. Executive Summary

**Key findings from this analysis:**

1. **Revenue concentration.** A minority of customers (Champions + Loyal Customers) generate a disproportionate share of revenue — standard Pareto behavior for e-commerce.
2. **High baseline churn.** The large majority of customers in this dataset never place a second order, which is a known characteristic of the Olist marketplace rather than a flaw in the model — see the leakage discussion in Notebook 07.
3. **Where to focus retention spend:** the "At Risk" and "Need Attention" segments contain customers who *have* purchased more than once but have gone quiet recently — these are the best return-on-investment win-back targets, more so than one-time buyers.
4. **Geography matters.** Revenue is concentrated in a handful of states, which can inform where to prioritize regional marketing or logistics investment.
5. **Next steps:** build a forward-looking (leakage-free) churn model using behavioral features (e.g. time-to-second-purchase, review sentiment, delivery experience) instead of a rule mirrored from Recency/Frequency, and connect the high-risk watchlist to an actual retention campaign to measure real-world lift.